In [ ]:
import os
import sys
from io import BytesIO

import numpy as np
import tensorflow as tf
from datasets import load_dataset
from PIL import Image
from sklearn import metrics

In [ ]:
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount('/content/drive')

    CACHE_DIR = '/content/drive/MyDrive/HuggingFace/cache'
    os.makedirs(CACHE_DIR, exist_ok=True)

    HF_TOKEN = userdata.get('HF_TOKEN')
    MODEL_PATH = '/content/drive/MyDrive/models/deforge_ela.keras'
else:
    from dotenv import load_dotenv

    CACHE_DIR = None

    load_dotenv()
    HF_TOKEN = os.getenv('HF_TOKEN')
    MODEL_PATH = os.path.join('models', 'deforge_ela.keras')

In [ ]:
IMAGE_SIZE = (224, 224)
NUM_CLASSES = 2
BATCH_SIZE = 128 if IN_COLAB else 32
MAX_SAMPLES = 1000  # Match RIGID evaluation cap per dataset

generator_mapping = {
    1: 'ADM',
    2: 'BigGAN',
    3: 'CycleGAN',
    4: 'DALLE2',
    5: 'GauGAN',
    6: 'Glide',
    7: 'Midjourney',
    8: 'ProGAN',
    9: 'SD14',
    10: 'SD15',
    11: 'SDXL',
    12: 'StarGAN',
    13: 'StyleGAN',
    14: 'StyleGAN2',
    15: 'VQDM',
    16: 'WhichFaceIsReal',
    17: 'Wukong',
}

In [ ]:
def ela_from_pil(pil_image: Image.Image, scale=(224, 224), quality=90) -> np.ndarray:
    """
    Performs Error Level Analysis (ELA) on a PIL image and returns a 3-channel RGB result.
    Must match exactly the preprocessing used during training.
    """
    image = pil_image.convert('RGB').resize(scale)

    buffer = BytesIO()
    image.save(buffer, format='JPEG', quality=quality)
    buffer.seek(0)

    compressed = Image.open(buffer)

    diff = np.abs(
        np.array(image, dtype=np.int16) - np.array(compressed, dtype=np.int16)
    )
    diff = np.clip(diff * 10, 0, 255).astype(np.uint8)

    return diff

In [ ]:
def calculate_auc_metrics(id_conf, ood_conf):
    """
    Calculate AUROC and FPR at 95% TPR for binary classification.

    Args:
        id_conf (np.ndarray): Confidence scores for ID (real) samples
        ood_conf (np.ndarray): Confidence scores for OOD (fake) samples

    Returns:
        tuple: (auroc, fpr_at_95_tpr)
    """
    all_conf = np.concatenate([id_conf, ood_conf])
    labels = np.concatenate([np.ones(len(id_conf)), np.zeros(len(ood_conf))])

    fpr, tpr, _ = metrics.roc_curve(labels, all_conf)
    auroc = metrics.auc(fpr, tpr)

    tpr_threshold = 0.95
    valid_indices = tpr >= tpr_threshold
    if np.any(valid_indices):
        fpr_at_95 = fpr[np.argmax(valid_indices)]
    else:
        fpr_at_95 = fpr[-1]
        print(f'Warning: 95% TPR not achievable. Max TPR: {tpr[-1]:.3f}')

    return auroc, fpr_at_95


def calculate_average_precision(id_predictions, ood_predictions):
    all_predictions = np.concatenate([id_predictions, ood_predictions])
    labels = np.concatenate(
        [np.ones(len(id_predictions)), np.zeros(len(ood_predictions))]
    )
    return metrics.average_precision_score(labels, all_predictions)


def sim_auc(similarities, datasets):
    """
    Calculate AUC and FPR95 for multiple OOD datasets against ID dataset.

    Args:
        similarities (list): List of confidence arrays, first one is ID (real)
        datasets (list): List of dataset names

    Returns:
        tuple: (average_auc, average_fpr95)
    """
    if len(similarities) != len(datasets):
        raise ValueError(
            'Number of similarity arrays must match number of dataset names'
        )
    if len(similarities) < 2:
        raise ValueError('At least 2 datasets (ID and OOD) are required')

    id_confi = similarities[0]
    auc_scores = []
    fpr_scores = []

    for ood_confi, dataset in zip(similarities[1:], datasets[1:]):
        auroc, fpr_95 = calculate_auc_metrics(id_confi, ood_confi)
        auc_scores.append(auroc)
        fpr_scores.append(fpr_95)
        print(f'Dataset: {dataset:<25} | AUC: {auroc:.4f} | FPR95: {fpr_95:.4f}')

    avg_auc = np.mean(auc_scores)
    avg_fpr = np.mean(fpr_scores)

    print('-' * 60)
    print(f'Average AUC: {avg_auc:.4f} | Average FPR95: {avg_fpr:.4f}')

    return avg_auc, avg_fpr


def sim_ap(similarities, datasets):
    """
    Calculate Average Precision for multiple OOD datasets against ID dataset.

    Args:
        similarities (list): List of confidence arrays, first one is ID (real)
        datasets (list): List of dataset names

    Returns:
        float: average AP score
    """
    if len(similarities) != len(datasets):
        raise ValueError(
            'Number of similarity arrays must match number of dataset names'
        )
    if len(similarities) < 2:
        raise ValueError('At least 2 datasets (ID and OOD) are required')

    id_confi = similarities[0]
    ap_scores = []

    for ood_confi, dataset in zip(similarities[1:], datasets[1:]):
        aver_p = calculate_average_precision(id_confi, ood_confi)
        ap_scores.append(aver_p)
        print(f'Dataset: {dataset:<25} | AP: {aver_p:.4f}')

    avg_ap = np.mean(ap_scores)
    print('-' * 40)
    print(f'Average AP: {avg_ap:.4f}')

    return avg_ap

In [ ]:
def get_confidence_scores(hf_subset, model, max_samples=MAX_SAMPLES):
    """
    Run ELA preprocessing and model inference on a HuggingFace dataset subset.
    Returns the model's confidence score for the 'real' class (index 0),
    which serves as the similarity signal — higher = more likely real (ID).

    Args:
        hf_subset: HuggingFace dataset split (already filtered)
        model: Loaded Keras model
        max_samples (int): Maximum number of samples to process

    Returns:
        np.ndarray: Array of real-class confidence scores
    """
    scores = []
    count = 0

    for sample in hf_subset:
        if count >= max_samples:
            break

        ela_img = ela_from_pil(sample['image'], scale=IMAGE_SIZE)
        x = ela_img.astype(np.float32) / 255.0
        x = np.expand_dims(x, axis=0)  # (1, H, W, 3)

        pred = model.predict(x, verbose=0)  # shape: (1, 2)
        # Index 0 = real, index 1 = fake (matches training label encoding)
        # Higher real-class score = more ID-like, consistent with RIGID's cosine similarity signal
        real_score = float(pred[0][0])
        scores.append(real_score)
        count += 1

    return np.array(scores)

In [ ]:
if IN_COLAB:
    print(f'Loading dataset. Colab detected: using Drive cache at {CACHE_DIR}')
else:
    print('Loading dataset. Local environment detected: using default HF cache.')

test_data = load_dataset(
    'TheKernel01/AIGC-Detection-Benchmark',
    split='test',
    token=HF_TOKEN,
    cache_dir=CACHE_DIR,
)

In [ ]:
print(f'Loading model from: {MODEL_PATH}')
model = tf.keras.models.load_model(MODEL_PATH)
model.summary()

In [ ]:
# Filter dataset into Real (ID) and per-generator Fake (OOD) subsets
all_generators = np.array(test_data['generator'])

# Real (ID) subset
real_indices = np.nonzero(all_generators == 0)[0].tolist()
real_hf = test_data.select(real_indices)

evaluation_datasets = [('Real (ID)', real_hf)]

# Per-generator fake (OOD) subsets
for gen_id, gen_name in generator_mapping.items():
    fake_indices = np.nonzero(all_generators == gen_id)[0].tolist()
    fake_hf = test_data.select(fake_indices)
    evaluation_datasets.append((f'{gen_name} (OOD)', fake_hf))

test_datasets = [name for name, _ in evaluation_datasets]

In [ ]:
# Run inference and collect confidence scores per dataset
confidence_datasets = []

for dataset_name, dataset_obj in evaluation_datasets:
    scores = get_confidence_scores(dataset_obj, model, max_samples=MAX_SAMPLES)

    if len(scores) > 0:
        print(
            f'{dataset_name:<25}, Image number: {len(scores)}, '
            f'mean confidence (real class): {scores.mean():.4f}'
        )
        confidence_datasets.append(scores)
    else:
        print(f'Warning: {dataset_name} is empty. Check your label filtering!')

In [ ]:
print('\n' + '=' * 60)
print('Detection Results AUC (Per Generator):')
print('=' * 60)
sim_auc(confidence_datasets, test_datasets)

print('\n' + '=' * 60)
print('Detection Results AP (Per Generator):')
print('=' * 60)
sim_ap(confidence_datasets, test_datasets)